In [ ]:
# Imports and NeuralProphet configurationimport osimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.metrics import mean_absolute_errorwarnings.filterwarnings('ignore')os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'from neuralprophet import NeuralProphet

# 3.3 NeuralProphet: Credit Reporting (South vs Non-South)

In [ ]:
# Load only required columns from the engineered full datasetcols = ['Product', 'region', 'year_quarter']df = pd.read_parquet('../data/processed/full_dataset.parquet', columns=cols)# Keep only Credit Reporting and create South vs Non-South segmentdf_cr = df[df['Product'].str.contains('Credit reporting', case=False, na=False)].copy()df_cr['segment'] = np.where(df_cr['region'].eq('South'), 'South', 'Non-South')# Aggregate to quarterly complaint counts by segmentquarterly_counts = (    df_cr.groupby(['year_quarter', 'segment'])    .size()    .unstack(fill_value=0)    .sort_index())quarterly_counts.index = pd.PeriodIndex(quarterly_counts.index, freq='Q')print(f'Filtered rows (Credit Reporting): {len(df_cr):,}')print('Quarterly shape:', quarterly_counts.shape)quarterly_counts.tail()

In [ ]:
# Visual check of the two segment seriesfig, ax = plt.subplots(figsize=(11, 4))for seg in ['South', 'Non-South']:    s = quarterly_counts[seg].copy()    s.index = s.index.to_timestamp(how='end')    ax.plot(s.index, s.values, label=seg)ax.set_title('Credit Reporting Complaints by Quarter: South vs Non-South')ax.set_ylabel('Complaint Count')ax.legend()plt.tight_layout()plt.show()

In [ ]:
# Train and evaluate NeuralProphet for each segmentdef split_series(ts):    n = len(ts)    test_size = max(4, int(round(n * 0.2)))    train = ts.iloc[:-test_size].copy()    test = ts.iloc[-test_size:].copy()    return train, testdef run_neuralprophet(ts):    train, test = split_series(ts)    train_df = pd.DataFrame({        'ds': train.index.to_timestamp(how='start'),        'y': train.values    })    test_df = pd.DataFrame({        'ds': test.index.to_timestamp(how='start'),        'y': test.values    })    model = NeuralProphet(        n_lags=4,        n_forecasts=1,        changepoints_range=0.9    )    model.fit(train_df, freq='QS', progress='off')    future = model.make_future_dataframe(train_df, periods=len(test_df))    fcst = model.predict(future)    yhat = fcst['yhat1'].tail(len(test_df)).values    mae = mean_absolute_error(test_df['y'].values, yhat)    return mae, test_df['y'].values, yhat

In [ ]:
# Execute per-segment runs and summarize holdout MAErows = []pred_store = {}for segment in ['South', 'Non-South']:    ts = quarterly_counts[segment]    mae, y_true, y_pred = run_neuralprophet(ts)    rows.append({'segment': segment, 'holdout_mae': float(mae)})    pred_store[segment] = {'y_true': y_true, 'y_pred': y_pred}np_summary = pd.DataFrame(rows)np_summary

In [ ]:
# Plot holdout actual vs predicted values for each segmentfig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)for i, seg in enumerate(['South', 'Non-South']):    y_true = pred_store[seg]['y_true']    y_pred = pred_store[seg]['y_pred']    axes[i].plot(y_true, label='Actual', marker='o')    axes[i].plot(y_pred, label='Predicted', marker='o')    axes[i].set_title(seg)    axes[i].legend()plt.tight_layout()plt.show()

## Notes- This notebook intentionally uses only Product, region, and year_quarter from the engineered full dataset.- Scope is strictly Credit Reporting with South vs Non-South segmentation.